# Retrain UniversalXAS head with E2E-head hyperparameters

This notebook uses the exported 64D features from `m3gnetAll8E2EUniversal/e2e_universal_seed42` and trains a fresh UniversalXAS MLP head with the same MLP-head hyperparameters used during the end-to-end run.

Flow:

1. load exported E2E-encoder features;
2. train a UniversalXAS head on all FEFF datasets;
3. fine-tune that head separately on each FEFF/VASP dataset using the same head hyperparameters;
4. compare stored/current eta values against this notebook's newly trained eta values.

No encoder retraining happens here.

## 1. Setup

In [1]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime
import json
import os

# Set before torch/lightning import if you want one GPU.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

from omnixas.data.ml_data import MLData, MLSplits
from omnixas.model.training import PlModule
from omnixas.model.xasblock import XASBlock
from omnixas.model.xasblock_regressor import XASBlockRegressor


def locate_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "omnixas").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the OmniXAS repo")


REPO_ROOT = locate_repo_root(Path.cwd().resolve())
FULLTRAINING_ROOT = Path(
    os.environ.get("OMNIXAS_FULL_TRAINING_ROOT", REPO_ROOT.parent / "fullTrainingCopy072726")
).expanduser().resolve()
SOURCE_RUN = FULLTRAINING_ROOT / "m3gnetAll8E2EUniversal" / "e2e_universal_seed42"
FEATURES_DIR = SOURCE_RUN / "features"

if not FEATURES_DIR.is_dir():
    raise FileNotFoundError(f"Missing exported E2E encoder features: {FEATURES_DIR}")

RUN_ID = f"e2e_head_hparams_retrain_{datetime.now():%Y%m%d_%H%M%S}"
RUN_DIR = SOURCE_RUN / "notebook_e2e_hparams_head_retrain" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)

FEFF_DATASETS = ["Ti_FEFF", "V_FEFF", "Cr_FEFF", "Mn_FEFF", "Fe_FEFF", "Co_FEFF", "Ni_FEFF", "Cu_FEFF"]
VASP_DATASETS = ["Ti_VASP", "Cu_VASP"]
DATASETS = FEFF_DATASETS + VASP_DATASETS

INPUT_DIM = 64
OUTPUT_DIM = 141
HIDDEN_DIMS = [500, 500, 550]
TASK_BATCH = {
    "Ti_FEFF": 32, "V_FEFF": 32, "Cr_FEFF": 32, "Mn_FEFF": 64,
    "Fe_FEFF": 64, "Co_FEFF": 32, "Ni_FEFF": 32, "Cu_FEFF": 32,
    "Ti_VASP": 64, "Cu_VASP": 64,
}
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(f"Repo: {REPO_ROOT}")
print(f"Source run: {SOURCE_RUN}")
print(f"Output run: {RUN_DIR}")
print(f"Device: {DEVICE}")

Repo: /mnt/c/Users/anton/desktop/omnixas
Source run: /mnt/c/Users/anton/desktop/fullTrainingCopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42
Output run: /mnt/c/Users/anton/desktop/fullTrainingCopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_e2e_hparams_head_retrain/e2e_head_hparams_retrain_20260729_092009
Device: cuda:0


## 2. Hyperparameters copied from the E2E head

The end-to-end run had one MLP head attached to the encoder. This notebook keeps that MLP architecture and head-training hyperparameters, but trains on already-exported 64D features.

In [2]:
settings = json.loads((SOURCE_RUN / "run_settings.json").read_text(encoding="utf-8"))["args"]

# E2E MLP-head hyperparameters.
SEED = int(settings["seed"])
HEAD_LR = float(settings["e2e_head_lr"])
HEAD_DROPOUT = float(settings["e2e_head_dropout"])
HEAD_EPOCHS = int(settings["e2e_epochs"])
HEAD_PATIENCE = 30  # same early-stopping patience used by the E2E trainer's encoder-stage callback
MIN_LR = 1e-6

hyperparams = pd.DataFrame([{
    "seed": SEED,
    "input_dim": INPUT_DIM,
    "hidden_dims": str(HIDDEN_DIMS),
    "output_dim": OUTPUT_DIM,
    "lr": HEAD_LR,
    "dropout": HEAD_DROPOUT,
    "max_epochs": HEAD_EPOCHS,
    "early_stopping_patience": HEAD_PATIENCE,
    "scheduler": "cosine",
    "cosine_t_max": HEAD_EPOCHS,
    "monitor": "val_median_mse",
}])
display(hyperparams)

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

,seed,input_dim,hidden_dims,output_dim,lr,dropout,max_epochs,early_stopping_patience,scheduler,cosine_t_max,monitor
0,42,64,"[500, 500, 550]",141,0.0007,0.25,300,30,cosine,300,val_median_mse


## 3. Load and validate exported features

In [3]:
def load_split(dataset: str) -> MLSplits:
    parts = {}
    for split_name in ["train", "val", "test"]:
        x_path = FEATURES_DIR / f"{dataset}_{split_name}_X.txt"
        y_path = FEATURES_DIR / f"{dataset}_{split_name}_y.txt"
        if not x_path.is_file() or not y_path.is_file():
            raise FileNotFoundError(f"Missing {dataset} {split_name}: {x_path}, {y_path}")
        X = np.atleast_2d(np.loadtxt(x_path, dtype=np.float32))
        y = np.atleast_2d(np.loadtxt(y_path, dtype=np.float32))
        if X.shape[0] != y.shape[0] or X.shape[1] != INPUT_DIM or y.shape[1] != OUTPUT_DIM:
            raise ValueError(f"Invalid {dataset} {split_name}: X={X.shape}, y={y.shape}")
        if not np.isfinite(X).all() or not np.isfinite(y).all():
            raise ValueError(f"Non-finite values in {dataset} {split_name}")
        parts[split_name] = MLData(X=X, y=y)
    return MLSplits(**parts)


def combine_splits(splits: list[MLSplits]) -> MLSplits:
    return MLSplits(
        train=MLData(X=np.concatenate([s.train.X for s in splits]), y=np.concatenate([s.train.y for s in splits])),
        val=MLData(X=np.concatenate([s.val.X for s in splits]), y=np.concatenate([s.val.y for s in splits])),
        test=MLData(X=np.concatenate([s.test.X for s in splits]), y=np.concatenate([s.test.y for s in splits])),
    )


splits = {dataset: load_split(dataset) for dataset in DATASETS}
universal_train_split = combine_splits([splits[dataset] for dataset in FEFF_DATASETS])

shape_rows = []
for dataset, split in splits.items():
    shape_rows.append({
        "dataset": dataset,
        "train": len(split.train),
        "val": len(split.val),
        "test": len(split.test),
        "X_dim": split.train.X.shape[1],
        "y_dim": split.train.y.shape[1],
    })
shape_table = pd.DataFrame(shape_rows)
display(shape_table)
print("Universal FEFF training rows:", len(universal_train_split.train))

,dataset,train,val,test,X_dim,y_dim
0,Ti_FEFF,5140,641,641,64,141
1,V_FEFF,8653,1080,1080,64,141
2,Cr_FEFF,2457,305,305,64,141
3,Mn_FEFF,13644,1704,1704,64,141
4,Fe_FEFF,9657,1205,1205,64,141
5,Co_FEFF,8605,1074,1074,64,141
6,Ni_FEFF,3471,432,432,64,141
7,Cu_FEFF,3340,416,416,64,141
8,Ti_VASP,3030,377,377,64,141
9,Cu_VASP,2584,321,321,64,141


Universal FEFF training rows: 54967


## 4. Train fresh UniversalXAS head

This trains on all eight FEFF datasets only, matching the UniversalXAS FEFF source-head setup. It is then evaluated on FEFF and VASP datasets.

In [4]:
def make_regressor(directory: Path, batch_size: int) -> XASBlockRegressor:
    XASBlock.DROPOUT = HEAD_DROPOUT
    return XASBlockRegressor(
        directory=str(directory),
        overwrite_save_dir=True,
        input_dim=INPUT_DIM,
        output_dim=OUTPUT_DIM,
        hidden_dims=HIDDEN_DIMS,
        batch_size=batch_size,
        max_epochs=HEAD_EPOCHS,
        early_stopping_patience=HEAD_PATIENCE,
        initial_lr=HEAD_LR,
        min_lr=MIN_LR,
        use_lr_finder=False,
        use_early_stopping=True,
        monitor_metric="val_median_mse",
        shuffle=True,
        lr_scheduler="cosine",
        cosine_t_max=HEAD_EPOCHS,
        cosine_eta_min=MIN_LR,
    )


universal_dir = RUN_DIR / "heads" / "universalXAS" / "All_FEFF" / "runs" / "e2e_head_hparams"
universal_regressor = make_regressor(universal_dir, batch_size=32)
universal_regressor.fit(universal_train_split)
universal_checkpoint = Path(universal_regressor.cfg.fetch_checkpoint("best"))
print("Universal checkpoint:", universal_checkpoint)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA GeForce RTX 4070 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /mnt/c/Users/anton/desktop/fullTrainingCopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_e2e_hparams_head_retrain/e2e_head_hparams_retrain_20260729_092009/heads/u

Sanity Checking: |                                                                                | 0/? [00:00…

/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.


Training: |                                                                                       | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

...

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

`Trainer.fit` stopped: `max_epochs=300` reached.
2026-07-29 09:49:05.426 | INFO     | omnixas.model.xasblock_regressor:fit:246 - Best models saved at /mnt/c/Users/anton/desktop/fullTrainingCopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_e2e_hparams_head_retrain/e2e_head_hparams_retrain_20260729_092009/heads/universalXAS/All_FEFF/runs/e2e_head_hparams/best-model-epoch=281-val_median_mse=0.0024.ckpt
2026-07-29 09:49:05.427 | INFO     | omnixas.model.xasblock_regressor:fit:247 - Best validation loss: 0.005062013864517212


Universal checkpoint: /mnt/c/Users/anton/desktop/fullTrainingCopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_e2e_hparams_head_retrain/e2e_head_hparams_retrain_20260729_092009/heads/universalXAS/All_FEFF/runs/e2e_head_hparams/best-model-epoch=281-val_median_mse=0.0024.ckpt


## 5. Evaluate UniversalXAS head

In [5]:
def predict_checkpoint(checkpoint: Path, X: np.ndarray, batch_size: int = 1024) -> np.ndarray:
    XASBlock.DROPOUT = HEAD_DROPOUT
    module = PlModule.load_from_checkpoint(
        checkpoint_path=str(checkpoint),
        model=XASBlock(INPUT_DIM, HIDDEN_DIMS, OUTPUT_DIM),
        lr=HEAD_LR,
    ).to(DEVICE).eval()
    loader = DataLoader(TensorDataset(torch.tensor(X, dtype=torch.float32)), batch_size=batch_size)
    batches = []
    with torch.inference_mode():
        for (batch,) in loader:
            batches.append(module(batch.to(DEVICE)).cpu().numpy())
    return np.concatenate(batches)


def eta_metrics(split: MLSplits, split_name: str, predictions: np.ndarray) -> dict[str, float]:
    target = getattr(split, split_name).y
    mse = np.mean((target - predictions) ** 2, axis=1)
    baseline = np.mean((target - split.train.y.mean(axis=0, keepdims=True)) ** 2, axis=1)
    median_mse = float(np.median(mse))
    baseline_median_mse = float(np.median(baseline))
    return {
        f"{split_name}_median_mse": median_mse,
        f"{split_name}_baseline_median_mse": baseline_median_mse,
        f"{split_name}_eta": baseline_median_mse / median_mse,
    }


universal_rows = []
for dataset in DATASETS:
    split = splits[dataset]
    predictions = predict_checkpoint(universal_checkpoint, split.test.X)
    row = {"dataset": dataset, "checkpoint": str(universal_checkpoint)}
    row.update(eta_metrics(split, "test", predictions))
    universal_rows.append(row)

notebook_universal_eval = pd.DataFrame(universal_rows)
notebook_universal_eval.to_csv(RUN_DIR / "notebook_universal_eval.csv", index=False)
display(notebook_universal_eval[["dataset", "test_median_mse", "test_baseline_median_mse", "test_eta"]].style.format({
    "test_median_mse": "{:.4f}", "test_baseline_median_mse": "{:.4f}", "test_eta": "{:.3f}",
}))

,dataset,test_median_mse,test_baseline_median_mse,test_eta
0,Ti_FEFF,0.0047,0.0486,10.254
1,V_FEFF,0.0049,0.0507,10.401
2,Cr_FEFF,0.0035,0.0563,16.166
3,Mn_FEFF,0.0019,0.0456,23.733
4,Fe_FEFF,0.0022,0.0288,13.307
5,Co_FEFF,0.0012,0.0300,24.315
6,Ni_FEFF,0.0017,0.0253,15.266
7,Cu_FEFF,0.0021,0.0128,6.116
8,Ti_VASP,0.1687,0.1707,1.012
9,Cu_VASP,0.0172,0.0229,1.336


## 6. Fine-tune Tuned-UniversalXAS heads with the same head hyperparameters

Each dataset gets one tuned head initialized from the fresh UniversalXAS checkpoint above. The hyperparameters stay fixed: `lr=7e-4`, `dropout=0.25`, cosine `T=300`, `max_epochs=300`, `patience=30`.

In [6]:
tuned_rows = []
for dataset in DATASETS:
    split = splits[dataset]
    out = RUN_DIR / "heads" / "tunedUniversalXAS" / dataset / "runs" / "e2e_head_hparams"

    tuned = make_regressor(universal_checkpoint.parent, batch_size=TASK_BATCH[dataset])
    tuned.cfg.overwrite_save_dir = False
    tuned.load("best")
    tuned.cfg.directory = str(out)
    tuned.cfg.overwrite_save_dir = True
    tuned.fit(split)

    tuned_checkpoint = Path(tuned.cfg.fetch_checkpoint("best"))
    predictions = predict_checkpoint(tuned_checkpoint, split.test.X)
    row = {"dataset": dataset, "checkpoint": str(tuned_checkpoint)}
    row.update(eta_metrics(split, "test", predictions))
    tuned_rows.append(row)
    print(f"{dataset}: tuned test_eta={row['test_eta']:.4f}")

notebook_tuned_eval = pd.DataFrame(tuned_rows)
notebook_tuned_eval.to_csv(RUN_DIR / "notebook_tuned_eval.csv", index=False)
display(notebook_tuned_eval[["dataset", "test_median_mse", "test_baseline_median_mse", "test_eta"]].style.format({
    "test_median_mse": "{:.4f}", "test_baseline_median_mse": "{:.4f}", "test_eta": "{:.3f}",
}))

2026-07-29 09:49:07.082 | INFO     | omnixas.model.xasblock_regressor:load:265 - Loading model from /mnt/c/Users/anton/desktop/fullTrainingCopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_e2e_hparams_head_retrain/e2e_head_hparams_retrain_20260729_092009/heads/universalXAS/All_FEFF/runs/e2e_head_hparams/best-model-epoch=281-val_median_mse=0.0024.ckpt
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /mnt/c/Users/anton/desktop/fullTrainingCopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_e2e_hparams_head_retrain/e2e_head_hparams_retrain_20260729_092009/heads/tunedUniv

Sanity Checking: |                                                                                | 0/? [00:00…

/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.


Training: |                                                                                       | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

...

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

`Trainer.fit` stopped: `max_epochs=300` reached.
2026-07-29 10:14:00.128 | INFO     | omnixas.model.xasblock_regressor:fit:246 - Best models saved at /mnt/c/Users/anton/desktop/fullTrainingCopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_e2e_hparams_head_retrain/e2e_head_hparams_retrain_20260729_092009/heads/tunedUniversalXAS/Cu_VASP/runs/e2e_head_hparams/best-model-epoch=285-val_median_mse=0.0030.ckpt
2026-07-29 10:14:00.129 | INFO     | omnixas.model.xasblock_regressor:fit:247 - Best validation loss: 0.004763023462146521


Cu_VASP: tuned test_eta=7.1281


,dataset,test_median_mse,test_baseline_median_mse,test_eta
0,Ti_FEFF,0.0047,0.0486,10.451
1,V_FEFF,0.0045,0.0507,11.279
2,Cr_FEFF,0.0031,0.0563,18.222
3,Mn_FEFF,0.0015,0.0456,29.935
4,Fe_FEFF,0.0020,0.0288,14.504
5,Co_FEFF,0.0011,0.0300,26.247
6,Ni_FEFF,0.0017,0.0253,14.997
7,Cu_FEFF,0.0018,0.0128,7.044
8,Ti_VASP,0.0330,0.1707,5.169
9,Cu_VASP,0.0032,0.0229,7.128


## 7. Compare current stored eta vs this notebook's eta

Current values come from the existing run's `universal_eval.csv` and `tuned_eval.csv`. Notebook values come from the fresh checkpoints trained above.

In [7]:
current_universal = pd.read_csv(SOURCE_RUN / "universal_eval.csv")[["dataset", "test_eta"]].rename(columns={"test_eta": "current_universal_eta"})
current_tuned = pd.read_csv(SOURCE_RUN / "tuned_eval.csv")[["dataset", "test_eta"]].rename(columns={"test_eta": "current_tuned_eta"})
new_universal = notebook_universal_eval[["dataset", "test_eta"]].rename(columns={"test_eta": "notebook_universal_eta"})
new_tuned = notebook_tuned_eval[["dataset", "test_eta"]].rename(columns={"test_eta": "notebook_tuned_eta"})

comparison = current_universal.merge(current_tuned, on="dataset", how="outer")
comparison = comparison.merge(new_universal, on="dataset", how="outer")
comparison = comparison.merge(new_tuned, on="dataset", how="outer")
comparison["delta_universal"] = comparison["notebook_universal_eta"] - comparison["current_universal_eta"]
comparison["delta_tuned"] = comparison["notebook_tuned_eta"] - comparison["current_tuned_eta"]
comparison = comparison.set_index("dataset").loc[DATASETS].reset_index()
comparison.to_csv(RUN_DIR / "current_vs_notebook_eta.csv", index=False)

cols = [
    "dataset",
    "current_universal_eta", "notebook_universal_eta", "delta_universal",
    "current_tuned_eta", "notebook_tuned_eta", "delta_tuned",
]
display(comparison[cols].style.format({column: "{:.3f}" for column in cols if column != "dataset"}))
print("Saved comparison:", RUN_DIR / "current_vs_notebook_eta.csv")

,dataset,current_universal_eta,notebook_universal_eta,delta_universal,current_tuned_eta,notebook_tuned_eta,delta_tuned
0,Ti_FEFF,10.214,10.254,0.040,10.606,10.451,-0.155
1,V_FEFF,10.448,10.401,-0.047,11.144,11.279,0.135
2,Cr_FEFF,15.018,16.166,1.149,17.372,18.222,0.850
3,Mn_FEFF,23.228,23.733,0.505,30.562,29.935,-0.627
4,Fe_FEFF,13.570,13.307,-0.263,14.288,14.504,0.216
5,Co_FEFF,22.568,24.315,1.748,25.102,26.247,1.145
6,Ni_FEFF,15.045,15.266,0.221,15.278,14.997,-0.281
7,Cu_FEFF,6.141,6.116,-0.025,6.751,7.044,0.293
8,Ti_VASP,1.032,1.012,-0.020,5.220,5.169,-0.050
9,Cu_VASP,1.393,1.336,-0.057,7.346,7.128,-0.218


Saved comparison: /mnt/c/Users/anton/desktop/fullTrainingCopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_e2e_hparams_head_retrain/e2e_head_hparams_retrain_20260729_092009/current_vs_notebook_eta.csv


## 8. Notes

- This notebook intentionally does **not** retrain the graph encoder.
- It uses the E2E encoder's exported 64D features and retrains only MLP heads.
- The final comparison is test eta; model selection during training still uses validation `val_median_mse` through checkpointing/early stopping.